# 4. Multi-Physics Basic Calculations and Statistical Analysis

This comprehensive tutorial demonstrates essential computational methods for analyzing multi-physics simulation data using MERA.jl. Learn to calculate fundamental quantities, statistical measures, and derived properties across hydro, particle, and clump datasets with proper unit handling and weighting schemes.

## Learning Objectives

By the end of this tutorial, you will be able to:

- **Calculate fundamental quantities** - Total mass, center-of-mass, and bulk velocities across all data types
- **Apply proper unit conversions** - Seamlessly work with physical units and automatic scaling
- **Perform statistical analysis** - Compute weighted and unweighted statistical measures
- **Extract derived quantities** - Use `getvar()` for predefined and custom variable calculations
- **Implement weighting schemes** - Apply mass, volume, and density weighting for accurate averages
- **Combine multi-physics data** - Joint calculations across hydro, particle, and clump datasets
- **Optimize computational workflows** - Efficient data processing and memory management

## Technical Foundation

### MERA Data Type Hierarchy

MERA organizes simulation data through a sophisticated type system that enables unified computational methods across different physics components:

**Core Data Types:**
- `ContainMassDataSetType` - Abstract supertype for mass-containing datasets
- `HydroDataType` - Hydrodynamic data with fluid properties (density, velocity, pressure)
- `PartDataType` - Particle data with discrete mass elements and positions
- `ClumpDataType` - Clump catalog data 
- `HydroPartType` - Combined hydro-particle data for mixed-physics analysis
![TypeHierarchy](./assets/TypeHierarchy.png)
**Unified Interface Benefits:**
- **Consistent function signatures** - Same functions work across all data types
- **Automatic unit handling** - Built-in scaling between code and physical units
- **Type-aware calculations** - Optimized algorithms for each data structure
- **Extensible framework** - Easy addition of new calculation methods

### Fundamental Calculation Functions

**Mass and Position Functions:**
- `msum(data, unit)` - Total mass calculation with automatic unit conversion
- `center_of_mass(data, unit)` / `com(data, unit)` - Mass-weighted center calculation
- `bulk_velocity(data, unit)` - Mass-weighted average velocity calculation
- `average_mweighted(data, var, unit)` - General mass-weighted averaging

**Statistical Analysis Functions:**
- `wstat(values, weights)` - Comprehensive weighted statistical analysis
- `getvar(data, vars, units)` - Variable extraction with unit conversion
- `getpositions(data, unit, center)` - Position extraction with coordinate transformation
- `getextent(data, unit, center)` - Domain boundary calculation

### Unit System Architecture

**Code Units (Default):**
- Internal simulation units optimized for numerical precision
- Dimensionless or normalized to characteristic scales
- Direct output from computational algorithms

**Physical Units (Converted):**
- Standard astronomical units (Msol, kpc, Myr, km/s, etc.)
- Automatic scaling using `info.scale` conversion factors
- User-specified through function parameters

**Conversion Hierarchy:**
```julia
# Manual scaling (explicit)
result_physical = result_code * info.scale.unit

# Automatic scaling (recommended)
result_physical = function(data, :unit) 
```

### Weighting Schemes

**Mass Weighting (Default):**
- Appropriate for most physical quantities
- Emphasizes high-mass regions
- Standard for velocity, position averaging

**Volume Weighting:**
- Used for density-related quantities
- Accounts for spatial resolution effects
- Important for grid-based hydrodynamic data

**No Weighting:**
- Simple arithmetic averaging
- Useful for discrete particle properties
- Equal treatment of all data elements

## Quick Reference

```julia
# Basic mass calculations
msum(gas, :Msol)                    # Total gas mass
msum([gas, particles], :Msol)       # Combined mass

# Center-of-mass calculations  
com(gas, :kpc)                      # Gas center-of-mass
com([gas, particles], :kpc)         # Joint center-of-mass

# Velocity analysis
bulk_velocity(gas, :km_s)           # Mass-weighted velocity
bulk_velocity(gas, :km_s, weighting=:volume)  # Volume-weighted

# Statistical analysis
wstat(getvar(gas, :rho, :g_cm3))    # Unweighted statistics
wstat(getvar(gas, :vx, :km_s), weight=getvar(gas, :mass))  # Weighted

# Variable extraction
getvar(gas, :mass, :Msol)           # Single variable
getvar(gas, [:mass, :ekin], [:Msol, :erg])  # Multiple variables

# Coordinate related
getpositions(gas, :kpc, center=[:boxcenter])  # Position arrays
getextent(gas, :kpc, center=[:boxcenter])     # Domain boundaries

# Time and scaling information
gettime(info, :Myr)                 # Simulation time
viewfields(info.scale)              # Available unit conversions
```

## Data Setup and Initialization

Load multi-physics simulation data for comprehensive analysis demonstrations:

In [1]:
# Example-data root. Point this at your own simulation folder, or set the
# MERA_EXAMPLES environment variable; every path below is built from it.
MERA_EXAMPLES = get(ENV, "MERA_EXAMPLES", "/Volumes/FASTStorage/Simulations/Mera-Tests");

using Mera
info = getinfo(400, "$MERA_EXAMPLES/RAMSES/manu_sim_sf_L14");
gas       = gethydro(info, [:rho, :vx, :vy, :vz], lmax=8); 
particles = getparticles(info, [:mass, :vx, :vy, :vz])
clumps    = getclumps(info);


*__   __ _______ ______   _______ 


|  |_|  |       |    _ | |   _   |
|       |    ___|   | || |  |_|  |
|       |   |___|   |_||_|       |
|       |    ___|    __  |       |
| ||_|| |   |___|   |  | |   _   |
|_|   |_|_______|___|  |_|__| |__|
Mera v1.8.0

[Mera]: 2026-08-06T10:34:42.146



Code: RAMSES
output [400] summary:
mtime: 

2018-09-05T09:51:55
ctime: 2025-06-29T20:06:45.267
simulation time: 594.98 [Myr]
boxlen: 48.0 [kpc]
ncpu: 2048
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 14 --> cellsize(s): 750.0 [pc] - 2.93 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  

7  --> (:rho, :vx, :vy, :vz, :p, :passive_scalar_1, :passive_scalar_2)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :thermal_pressure, :passive_scalar_1, :passive_scalar_2)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Npart:    5.091500e+05 
- Nstars:   5.066030e+05 


...

Combining results from 4 thread(s)...
Found 5.089390e+05 particles
Memory used for data table :

31.064148902893066 MB
-------------------------------------------------------

[Mera]: Get clump data: 2026-08-06T10:35:10.082



domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

Read 12 colums: 


[:index, :lev, :parent, :ncell, :peak_x, :peak_y, :peak_z, Symbol("rho-"), Symbol("rho+"), :rho_av, :mass_cl, :relevance]
Memory used for data table :

61.58203125 KB
-------------------------------------------------------



### Unit Conversion System

MERA provides comprehensive unit conversion capabilities through the `info.scale` object, which contains conversion factors between code units and physical units. Understanding this system is crucial for accurate scientific analysis.

**Core Conversion Principles:**
- **Code units** - Internal simulation units optimized for numerical stability
- **Physical units** - Standard astronomical units for scientific interpretation
- **Automatic scaling** - Functions handle conversions internally when units are specified
- **Consistent framework** - Same unit system across all data types and functions

Many functions can provide results in selected units through automatic internal scaling:

In [2]:
viewfields(info.scale)


[Mera]: Fields to scale from user/code units to selected units


Mpc	= 0.0010000000000006482
kpc	= 1.0000000000006481
pc	= 1000.0000000006482
mpc	= 1.0000000000006482e6
ly	= 3261.5637769461323
Au	= 2.0626480623310105e23
km	= 3.0856775812820004e16
m	= 3.085677581282e19
cm	= 3.085677581282e21
mm	= 3.085677581282e22
μm	= 3.085677581282e25
Mpc3	= 1.0000000000019446e-9
kpc3	= 1.0000000000019444
pc3	= 1.0000000000019448e9
mpc3	= 1.0000000000019446e18
ly3	= 3.469585750743794e10
Au3	= 8.775571306099254e69
km3	= 2.9379989454983075e49
m3	= 2.9379989454983063e58
cm3	= 2.9379989454983065e64
mm3	= 2.937998945498306e67
μm3	= 2.937998945498306e76
Msol_pc3	= 0.9997234790001649
Msun_pc3	= 0.9997234790001649
g_cm3	= 6.76838218451376e-23
Msol_pc2	= 999.7234790008131
Msun_pc2	= 999.7234790008131
g_cm2	= 0.20885045168302602
Gyr	= 0.014910986463557083
Myr	= 14.910986463557084
yr	= 1.4910986463557083e7
s	= 4.70554946422349e14
ms	= 4.70554946422349e17
Msol	= 9.99723479002109e8
Msun	= 9.99723479002109e8
Mearth	= 3.329677459032007e14
Mjupiter	= 1.0476363431814971e12
g	= 1.98

## Total Mass Calculations

Mass calculations form the foundation of astrophysical analysis, providing essential information about the distribution of matter across different simulation components. MERA's `msum()` function offers sophisticated mass calculation capabilities with automatic unit conversion and support for multi-physics datasets.

### Key Features:
- **Universal data type support** - Works with hydro, particle, and clump data
- **Automatic unit conversion** - Built-in scaling to physical units
- **Multi-dataset combinations** - Joint mass calculations across data types
- **Precision handling** - Optimized algorithms for numerical accuracy

### Physical Basis:
- **Hydrodynamic mass** - Derived from density and cell volume (ρ × V)
- **Particle mass** - Direct summation of discrete particle masses
- **Clump mass** - Hierarchical structure mass accounting

### Basic Mass Calculation

The `msum()` function calculates the total mass of data assigned to any MERA object. For hydrodynamic data, mass is derived from density and cell-size (level) of all elements, while particle data uses direct mass summation.

**Manual Unit Conversion:**
The traditional approach requires manual scaling using `info.scale.Msol` (or equivalent):

In [3]:
println( "Gas Mtot:       ", msum(gas)       * info.scale.Msol, " Msol" )
println( "Particles Mtot: ", msum(particles) * info.scale.Msol, " Msol" )
println( "Clumps Mtot:    ", msum(clumps)    * info.scale.Msol, " Msol" )

Gas Mtot:       2.6703951073850353e10

 Msol
Particles Mtot: 5.804426008528429e9

 Msol
Clumps Mtot:    1.3743280681841675e10

 Msol


### Automatic Unit Conversion

**Recommended Approach:**
The modern approach uses built-in unit conversion by providing a unit argument directly to the function:

In [4]:
println( "Gas Mtot:       ", msum(gas, :Msol)       , " Msol" )
println( "Particles Mtot: ", msum(particles, :Msol) , " Msol" )
println( "Clumps Mtot:    ", msum(clumps, :Msol)    , " Msol" )

Gas Mtot:       2.6703951073850353e10 Msol
Particles Mtot: 5.804426008528429e9 Msol
Clumps Mtot:    1.3743280681841675e10 Msol


The following methods are defined on the function `msum`:

In [5]:
methods(msum)

# 2 methods for generic function "msum" from Mera:
 [1] msum(dataobject::ContainMassDataSetType, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:63
 [2] msum(dataobject::ContainMassDataSetType; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:67

## Center-Of-Mass
The function `center_of_mass` or `com` calculates the center-of-mass of the data that is assigned to the provided object.

In [6]:
println( "Gas COM:       ", center_of_mass(gas)       .* info.scale.kpc, " kpc" )
println( "Particles COM: ", center_of_mass(particles) .* info.scale.kpc, " kpc" )
println( "Clumps COM:    ", center_of_mass(clumps)    .* info.scale.kpc, " kpc" );

Gas COM:       (

23.258011243936238, 23.76594380898452, 23.972244037494438) kpc
Particles COM: (22.891354761211396, 24.17414728268034, 24.003205056545642) kpc
Clumps COM:    (23.135765457064572, 23.741712325649264, 24.0050127185862) kpc


The units for the results can be calculated by the function itself by providing a unit-argument:

In [7]:
println( "Gas COM:       ", center_of_mass(gas, :kpc)       , " kpc" )
println( "Particles COM: ", center_of_mass(particles, :kpc) , " kpc" )
println( "Clumps COM:    ", center_of_mass(clumps, :kpc)    , " kpc" );

Gas COM:       (

23.258011243936238, 23.76594380898452, 23.972244037494438) kpc
Particles COM: (22.891354761211396, 24.17414728268034, 24.003205056545642) kpc
Clumps COM:    (23.135765457064572, 23.741712325649264, 24.0050127185862) kpc


A shorter name for the function `center_of_mass` is defined as `com` :

In [8]:
println( "Gas COM:       ", com(gas, :kpc)       , " kpc" )
println( "Particles COM: ", com(particles, :kpc) , " kpc" )
println( "Clumps COM:    ", com(clumps, :kpc)    , " kpc" );

Gas COM:       (

23.258011243936238, 23.76594380898452, 23.972244037494438) kpc
Particles COM: (22.891354761211396, 24.17414728268034, 24.003205056545642) kpc
Clumps COM:    (23.135765457064572, 23.741712325649264, 24.0050127185862) kpc


The result of the coordinates (x, y, z) can be assigned e.g. to a tuple or to three single variables:

In [9]:
# return coordinates in a tuple
com_gas = com(gas, :kpc)
println( "Tuple:      ", com_gas, " kpc" )

# return coordinates into variables
x_pos, y_pos, z_pos = com(gas, :kpc);  #create variables
println("Single vars: ", x_pos, "  ", y_pos, "  ", z_pos, "  kpc")

Tuple:      (23.258011243936238, 23.76594380898452, 23.972244037494438) kpc
Single vars: 23.258011243936238  23.76594380898452  23.972244037494438  kpc


Calculate the joint centre-of-mass from the hydro and particle data. Provide the hydro and particle data with an array (independent order):

In [10]:
println( "Joint COM (Gas + Particles): ", center_of_mass([gas,particles], :kpc) , " kpc" )
println( "Joint COM (Particles + Gas): ", center_of_mass([particles,gas], :kpc) , " kpc" )

Joint COM (Gas + Particles): (

23.192544105902943, 23.83882923383144, 23.9777721805675) kpc
Joint COM (Particles + Gas): (

23.19254410544675, 23.838829233362418, 23.977772180095187) kpc


Use the shorter name `com` that is defined as the function `center_of_mass` :

In [11]:
println( "Joint COM (Gas + Particles): ", com([gas,particles], :kpc) , " kpc" )
println( "Joint COM (Particles + Gas): ", com([particles,gas], :kpc) , " kpc" )

Joint COM (Gas + Particles): (

23.192544105902943, 23.83882923383144, 23.9777721805675) kpc
Joint COM (Particles + Gas): (

23.19254410544675, 23.838829233362418, 23.977772180095187) kpc


In [12]:
methods(center_of_mass)

# 4 methods for generic function "center_of_mass" from Mera:
 [1] center_of_mass(dataobject::Vector{HydroPartType}, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:242
 [2] center_of_mass(dataobject::Vector{HydroPartType}; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:246
 [3] center_of_mass(dataobject::ContainMassDataSetType, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:124
 [4] center_of_mass(dataobject::ContainMassDataSetType; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:128

In [13]:
methods(com)

# 4 methods for generic function "com" from Mera:
 [1] com(dataobject::Vector{HydroPartType}, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:274
 [2] com(dataobject::Vector{HydroPartType}; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:278
 [3] com(dataobject::ContainMassDataSetType, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:157
 [4] com(dataobject::ContainMassDataSetType; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:161

## Bulk Velocity

The function `bulk_velocity` or `average_velocity` calculates the average velocity (with and without mass-weight) of the data that is assigned to the provided object. It can also be used for the clump data if it has velocity components: vx, vy, vz. The default is with mass-weighting:

In [14]:
println( "Gas:       ", bulk_velocity(gas, :km_s)       , " km/s" )
println( "Particles: ", bulk_velocity(particles, :km_s) , " km/s" )

Gas:       (-1.441830310542467, -11.708719305767854, -0.5393243496862989) km/s
Particles: (-11.623422700314567, -18.440572802490294, -0.32919277314175355) km/s


In [15]:
println( "Gas:       ", average_velocity(gas, :km_s)       , " km/s" )
println( "Particles: ", average_velocity(particles, :km_s) , " km/s" )

Gas:       (-1.441830310542467, -11.708719305767854, -0.5393243496862989) km/s
Particles: (-11.623422700314567, -18.440572802490294, -0.32919277314175355) km/s


Without mass-weighting:
- gas: volume or :no weighting 
- particles: no weighting

In [16]:
println( "Gas:       ", bulk_velocity(gas, :km_s, weighting=:volume)       , " km/s" )
println( "Particles: ", bulk_velocity(particles, :km_s, weighting=:no) , " km/s" )

Gas:       (

1.5248458901822848, -8.770913864354457, -0.5037635305158429) km/s
Particles: (-11.594477384589647, -18.38859118719373, -0.3097746295267971) km/s


In [17]:
println( "Gas:       ", average_velocity(gas, :km_s, weighting=:volume)       , " km/s" )
println( "Particles: ", average_velocity(particles, :km_s, weighting=:no) , " km/s" )

Gas:       (1.5248458901822848, -8.770913864354457, -0.5037635305158429) km/s
Particles: (-11.594477384589647, -18.38859118719373, -0.3097746295267971) km/s


In [18]:
methods(bulk_velocity)

# 2 methods for generic function "bulk_velocity" from Mera:
 [1] bulk_velocity(dataobject::ContainMassDataSetType, unit::Symbol; weighting, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:415
 [2] bulk_velocity(dataobject::ContainMassDataSetType; unit, weighting, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:420

In [19]:
methods(average_velocity)

# 2 methods for generic function "average_velocity" from Mera:
 [1] average_velocity(dataobject::ContainMassDataSetType, unit::Symbol; weighting, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:447
 [2] average_velocity(dataobject::ContainMassDataSetType; unit, weighting, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:451

## Mass Weighted Average
The functions `center_of_mass` and `bulk_velocity` use the function `average_mweighted` (average_mass-weighted) in the backend which can be feeded with any kind of variable that is pre-defined for the `getvar()` function or exists in the datatable. See the defined method and at getvar() below:

In [20]:
methods( average_mweighted )

# 1 method for generic function "average_mweighted" from Mera:
 [1] average_mweighted(dataobject::ContainMassDataSetType, var::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:316

<a id="Statistics"></a>

## Get Predefined Quantities
Here, we only show the examples with the hydro-data:

In [21]:
info = getinfo(1, "$MERA_EXAMPLES/RAMSES/manu_stable_2019", verbose=false);
gas = gethydro(info, [:rho, :vx, :vy, :vz], verbose=false); 

Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   6%|███▏                                              |  ETA: 0:00:17 ( 0.56  s/it)

Processing files:   9%|████▊                                             |  ETA: 0:00:18 ( 0.62  s/it)

Processing files:  12%|██████▎                                           |  ETA: 0:00:15 ( 0.54  s/it)

Processing files:  44%|█████████████████████▉                            |  ETA: 0:00:08 ( 0.47  s/it)

...

Processing files:  88%|███████████████████████████████████████████▊      |  ETA: 0:00:02 ( 0.51  s/it)

Processing files:  91%|█████████████████████████████████████████████▎    |  ETA: 0:00:01 ( 0.49  s/it)

Processing files:  94%|██████████████████████████████████████████████▉   |  ETA: 0:00:01 ( 0.48  s/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:14 ( 0.45  s/it)



✓ File processing complete! Combining results...


Use `getvar` to extract variables or derive predefined quantities from the database, dependent on the data type.
See the possible variables:

In [22]:
getvar()

Predefined vars that can be calculated for each cell/particle:
----------------------------------------------------------------
=============================[gas]:=============================
       -all the non derived hydro vars-
:cpu, :level, :rho, :cx, :cy, :cz, :vx, :vy, :vz, :p, var6,...

              -derived hydro vars-
:x, :y, :z
:mass, :cellsize, :volume, :freefall_time
:cs, :mach, :machx, :machy, :machz, :jeanslength, :jeansnumber, :jeansmass
:virial_parameter_local
:T, :Temp, :Temperature with p/rho
:etherm (thermal energy per cell)
:overdensity, :delta (gas overdensity ρ/ρ̄_b−1; cosmological runs only)

:entropy_specific (specific entropy)
:entropy_index (dimensionless adiabatic constant)
:entropy_density (entropy per unit volume)
:entropy_per_particle (entropy per particle)
:entropy_total (total entropy per cell/particle)

          -magnetohydrodynamic Mach numbers-
:mach_alfven, :mach_fast, :mach_slow

==========================[particles]:==========================
 

### Get a Single Quantity
In the following example, we calculate the mass for each cell of the hydro data. 
- The output is a 1dim array in code units by default (mass1).
- Each element/cell can be scaled to Msol units by the elementwise multiplikation **gas.scale.Msol** (mass2). 
- The `getvar` function supports intrinsic scaling to a selected unit (mass3).
- The selected unit does not need a keyword argument if the following order is maintained: dataobject, variable, unit

In [23]:
mass1 = getvar(gas, :mass) # [code units]
mass2 = getvar(gas, :mass) * gas.scale.Msol # scale the result (1dim array) from code units to solar masses
mass3 = getvar(gas, :mass, unit=:Msol) # unit calculation, provided by a keyword argument [Msol]
mass4 = getvar(gas, :mass, :Msol) # unit calculation provided by an argument [Msol]

# construct a three dimensional array to compare the three created arrays column wise:  
mass_overview = [mass1 mass2 mass3 mass4] 

37898393×4 Matrix{Float64}:
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 8.9407e-7   894.07     894.07     894.07
 ⋮                                 
 1.02889e-7  102.889    102.889    102.889
 1.02889e-7  102.889    102.889    102.889
 1.94423e-7  194.423    194.423    194.423
 1.94423e-7  194.423    194.423    194.423
 8.90454e-8   89.0454    89.0454    89.0454
 8.90454e-8   89.0454    89.0454    89.0454
 2.27641e-8   22.7641    22.7641    22.7641
 2.27641e-8   22.7641    22.7641    22.7641
 8.42157e-9    8.42157    8.42157    8.421

Furthermore, we provide a simple function to get the mass of each cell in code units:

In [24]:
mass_cells = getmass(gas); # [code units]


### Available Methods

To see all available methods for the `msum` function, we can use Julia's introspection:

In [25]:
methods(msum)

# 2 methods for generic function "msum" from Mera:
 [1] msum(dataobject::ContainMassDataSetType, unit::Symbol; mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:63
 [2] msum(dataobject::ContainMassDataSetType; unit, mask)
     @ ~/code-github/Mera.jl/src/functions/basic_calc.jl:67

### Get Multiple Quantities
Get several quantities with one function call by passing an array containing the selected variables. 
`getvar` returns a dictionary containing 1dim arrays for each quantity in code units:

In [26]:
# array of variables -> Dict of 1-dim arrays, in code units
quantities = getvar(gas, [:mass, :ekin])


Dict{Any, Any} with 2 entries:
  :mass => [8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407e-7, 8…
  :ekin => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  2.28274e-7, 2.…

The units for each quantity can by passed as an array to the keyword argument "units" (plural, compare with single quantitiy call above) by preserving the order of the vars argument:

In [27]:
# units (plural) takes one unit per requested quantity
quantities = getvar(gas, [:mass, :ekin], units=[:Msol, :erg])


Dict{Any, Any} with 2 entries:
  :mass => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894…
  :ekin => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.95354e49, 1.…

The function can be called without any keywords by preserving the following order: dataobject, variables, units

In [28]:
quantities = getvar(gas, [:mass, :ekin], [:Msol, :erg])

Dict{Any, Any} with 2 entries:
  :mass => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894…
  :ekin => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  1.95354e49, 1.…

The arrays of the single quantities can be accessed from the dictionary:

In [29]:
# each quantity is one entry of the returned dictionary
quantities[:mass]


37898393-element Vector{Float64}:
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
 894.0696716308591
   ⋮
 102.88910576564386
 102.88910576564386
 194.42336261293337
 194.42336261293337
  89.04538915743468
  89.04538915743468
  22.764121923068824
  22.764121923068824
   8.421571563820482
   8.421571563820482
  36.50851622718897
  36.50851622718897

If all selected variables should be of the same unit use the following arguments: dataobject, array of quantities, unit (no array needed):

In [30]:
# one unit for every quantity: dataobject, variables, unit
velocities = getvar(gas, [:vx, :vy, :vz], :km_s)


Dict{Any, Any} with 3 entries:
  :vy => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  -97.5301, -97.53…
  :vz => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0…
  :vx => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  -24.307, -24.307…

### Get Quantities related to a center

Some quantities are related to a given center, e.g. radius in cylindrical coordinates, see the overview :

In [31]:
getvar()

Predefined vars that can be calculated for each cell/particle:
----------------------------------------------------------------
=============================[gas]:=============================
       -all the non derived hydro vars-
:cpu, :level, :rho, :cx, :cy, :cz, :vx, :vy, :vz, :p, var6,...

              -derived hydro vars-
:x, :y, :z
:mass, :cellsize, :volume, :freefall_time
:cs, :mach, :machx, :machy, :machz, :jeanslength, :jeansnumber, :jeansmass
:virial_parameter_local
:T, :Temp, :Temperature with p/rho
:etherm (thermal energy per cell)
:overdensity, :delta (gas overdensity ρ/ρ̄_b−1; cosmological runs only)

:entropy_specific (specific entropy)
:entropy_index (dimensionless adiabatic constant)
:entropy_density (entropy per unit volume)
:entropy_per_particle (entropy per particle)
:entropy_total (total entropy per cell/particle)

          -magnetohydrodynamic Mach numbers-
:mach_alfven, :mach_fast, :mach_slow

==========================[particles]:==========================
 

     -angular momentum-
:l, :lx, :ly, :lz (Cartesian components)
:lr_cylinder, :lϕ_cylinder (cylindrical components)
:lr_sphere, :lθ_sphere, :lϕ_sphere (spherical components)

     -cylindrical acceleration components, gravity-
:ar_cylinder, :aϕ_cylinder

     -spherical acceleration components, gravity-
:ar_sphere, :aθ_sphere, :aϕ_sphere
----------------------------------------------------------------


The unit of the provided center-array (in cartesian coordinates: x,y.z) is given by the keyword argument `center_unit` (default: code units).
The function returns the quantitites in code units:

In [32]:
cv = (gas.boxlen / 2.) * gas.scale.kpc # provide the box-center in kpc
# e.g. for :mass the center keyword is ignored
quantities = getvar(gas, [:mass, :r_cylinder], center=[cv, cv, cv], center_unit=:kpc) 

Dict{Any, Any} with 2 entries:
  :r_cylinder => [70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345…
  :mass       => [8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407e-7, 8.9407…

Here, the function returns the result in the units that are provided. Note: E.g. the quantities :mass and :v (velocity) are not affected by the given center.

In [33]:
quantities = getvar(gas, [:mass, :r_cylinder, :v], units=[:Msol, :kpc, :km_s], center=[cv, cv, cv], center_unit=:kpc)

Dict{Any, Any} with 3 entries:
  :r_cylinder => [70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345…
  :v          => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  100.513,…
  :mass       => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.0…

Use the short notation for the box center :bc or :boxcenter for all dimensions (x,y,z). In this case the keyword `center_unit` is ignored:

In [34]:
quantities = getvar(gas, [:mass, :r_cylinder, :v], units=[:Msol, :kpc, :km_s], center=[:boxcenter])

Dict{Any, Any} with 3 entries:
  :r_cylinder => [70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345…
  :v          => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  100.513,…
  :mass       => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.0…

In [35]:
quantities = getvar(gas, [:mass, :r_cylinder, :v], units=[:Msol, :kpc, :km_s], center=[:bc])

Dict{Any, Any} with 3 entries:
  :r_cylinder => [70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345, 70.4345…
  :v          => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  100.513,…
  :mass       => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.0…

Use the box center notation for individual dimensions, here x,z. The keyword `center_unit` is needed for the y-coordinates:

In [36]:
quantities = getvar(gas, [:mass, :r_cylinder, :v], units=[:Msol, :kpc, :km_s], center=[:bc, 24., :bc], center_unit=:kpc)

Dict{Any, Any} with 3 entries:
  :r_cylinder => [55.2012, 55.2012, 55.2012, 55.2012, 55.2012, 55.2012, 55.2012…
  :v          => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  100.513,…
  :mass       => [894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.07, 894.0…

## Create Costum Quantities

**Example1:** Represent the positions of the data as the radius for a disk, centred in the simulation box (cylindrical coordinates):

In [37]:
boxlen = info.boxlen
cv = boxlen / 2. # box-center
levels = getvar(gas, :level) # get the level of each cell
cellsize = boxlen ./ 2 .^levels # calculate the cellsize for each cell (code units)

# or use the predefined quantity
cellsize = getvar(gas, :cellsize)


# convert the cell-number (related to the levels) into positions (code units), relative to the box center
x = getvar(gas, :cx) .* cellsize .- cv # (code units)
y = getvar(gas, :cy) .* cellsize .- cv # (code units)

# or use the predefined quantity
x = getvar(gas, :x, center=[:bc])
y = getvar(gas, :y, center=[:bc])


# calculate the cylindrical radius and scale from code units to kpc
radius = sqrt.(x.^2 .+ y.^2) .* info.scale.kpc

37898393-element Vector{Float64}:
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
 70.4344645322994
  ⋮
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174
 20.049876962806174

### Use IndexedTables Functions
see <https://juliadb.juliadata.org/stable/>

In [38]:
using Mera.IndexedTables

Example: Get the mass for each gas cell:
m_i  = ρ_i * cell_volume_i = ρ_i * (boxlen / 2^level)^3

#### Version 1
Use the `select` function and calculate the mass for each cell:

In [39]:
boxlen = gas.boxlen
level = select(gas.data, :level ) # get level information from each cell
cellvol = (boxlen ./ 2 .^level).^3 # calculate volume for each cell
mass1 = select(gas.data, :rho) .* cellvol .* info.scale.Msol; # calculate the mass for each cell in Msol units

#### Version 2
Use a single time the `select` function to do the calculations from above :

In [40]:
mass2 = select( gas.data, (:rho, :level)=>p->p.rho * (boxlen / 2^p.level)^3 ) .* info.scale.Msol;

#### Version 3
Use the `map` function to do the calculations from above :

In [41]:
mass3 = map(p->p.rho * (boxlen / 2^p.level)^3, gas.data) .* info.scale.Msol;

Comparison of the results:

In [42]:
[mass1 mass2 mass3]

37898393×3 Matrix{Float64}:
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
 894.07     894.07     894.07
   ⋮                   
 102.889    102.889    102.889
 102.889    102.889    102.889
 194.423    194.423    194.423
 194.423    194.423    194.423
  89.0454    89.0454    89.0454
  89.0454    89.0454    89.0454
  22.7641    22.7641    22.7641
  22.7641    22.7641    22.7641
   8.42157    8.42157    8.42157
   8.42157    8.42157    8.42157
  36.5085    36.5085    36.5085
  36.5085    36.5085    36.5085

## Statistical Analysis

Statistical analysis provides essential insights into the distribution and characteristics of simulation data. MERA's `wstat` function offers comprehensive statistical calculations with support for both unweighted and mass-weighted analysis across all data types.

### Key Features

- **Comprehensive Statistics**: Mean, median, standard deviation, min/max, quartiles, and more
- **Weighted Analysis**: Mass-weighted, volume-weighted, or custom weighting schemes
- **Multi-Physics Support**: Consistent interface across hydro, particle, and clump data
- **Memory Efficient**: Optimized calculations for large datasets
- **Physical Units**: Automatic unit conversion for all statistical quantities

### Statistical Quantities Available

The `wstat` function returns a structured object containing:
- **mean**: Arithmetic or weighted mean
- **median**: 50th percentile value
- **std**: Standard deviation
- **min/max**: Extreme values
- **q25/q75**: 25th and 75th percentiles
- **count**: Number of data points

### Quick Reference

```julia
# Unweighted statistics
stats = wstat(getvar(data, :variable, :unit))

# Weighted statistics
stats = wstat(getvar(data, :variable, :unit), weight=getvar(data, :mass))

# Access results
println("Mean: ", stats.mean)
println("Std:  ", stats.std)
println("Range: ", stats.min, " to ", stats.max)
```

In [43]:
info = getinfo(400, "$MERA_EXAMPLES/RAMSES/manu_sim_sf_L14", verbose=false);
gas       = gethydro(info, [:rho, :vx, :vy, :vz], lmax=8, smallr=1e-5, verbose=false); 
particles = getparticles(info, [:mass, :vx, :vy, :vz], verbose=false)
clumps    = getclumps(info, verbose=false);

Processing files:   0%|▎                                                 |  ETA: 0:00:29 (14.23 ms/it)

Processing files:   1%|▋                                                 |  ETA: 0:00:19 ( 9.46 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:19 ( 9.63 ms/it)

Processing files:   3%|█▎                                                |  ETA: 0:00:19 ( 9.40 ms/it)

Processing files:   3%|█▋                                                |  ETA: 0:00:18 ( 9.00 ms/it)

...

Processing files:  99%|█████████████████████████████████████████████████▎|  ETA: 0:00:00 ( 8.87 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▋|  ETA: 0:00:00 ( 8.87 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▉|  ETA: 0:00:00 ( 8.88 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:18 ( 8.88 ms/it)



✓ File processing complete! Combining results...


Pass any kind of Array{<:Real,1} (Float, Integer,...) to the `wstat` function to get several unweighted statistical quantities at once:

In [44]:
stats_gas       = wstat( getvar(gas,       :vx,     :km_s)     )
stats_particles = wstat( getvar(particles, :vx,     :km_s)     )
stats_clumps    = wstat( getvar(clumps,    :rho_av, :Msol_pc3) );

The result is an object that contains several fields with the statistical quantities:

In [45]:
println( typeof(stats_gas) )
println( typeof(stats_particles) )
println( typeof(stats_clumps) )
propertynames(stats_gas)

Mera.

WStatType
Mera.WStatType
Mera.WStatType


(:mean, :median, :std, :skewness, :kurtosis, :min, :max)

In [46]:
println( "Gas        <vx>_allcells     : ",  stats_gas.mean,       " km/s" )
println( "Particles  <vx>_allparticles : ",  stats_particles.mean, " km/s" )
println( "Clumps <rho_av>_allclumps    : ",  stats_clumps.mean,    " Msol/pc^3" )

Gas        <vx>_allcells     : -2.931877465071372 km/s
Particles  <vx>_allparticles : -11.594477384589647 km/s
Clumps <rho_av>_allclumps    : 594.7315900915924 Msol/pc^3


In [47]:
println( "Gas        min/max_allcells     : ",  stats_gas.min,      "/", stats_gas.max,       " km/s" )
println( "Particles  min/max_allparticles : ",  stats_particles.min,"/", stats_particles.max, " km/s" )
println( "Clumps     min/max_allclumps    : ",  stats_clumps.min,   "/", stats_clumps.max,    " Msol/pc^3" )

Gas        min/max_allcells     : -676.5464963488397/894.9181733956399 km/s
Particles  min/max_allparticles : -874.6440509326601/670.7956741234592 km/s
Clumps     min/max_allclumps    : 125.4809686796669/5357.370234867635 Msol/pc^3


## Weighted Statistics
Pass any kind of Array{<:Real,1} (Float, Integer,...) for the given variables and one for the weighting with the same length. The weighting goes cell by cell, particle by particle, clump by clump, etc...:

In [48]:
stats_gas       = wstat( getvar(gas,       :vx,     :km_s), weight=getvar(gas,       :mass  ));
stats_particles = wstat( getvar(particles, :vx,     :km_s), weight=getvar(particles, :mass   ));
stats_clumps    = wstat( getvar(clumps,    :peak_x, :kpc ), weight=getvar(clumps,    :mass_cl))  ;

Without the keyword `weight` the following order for the given arrays has to be maintained: values, weight

In [49]:
stats_gas       = wstat( getvar(gas,       :vx,     :km_s), getvar(gas,       :mass  ));
stats_particles = wstat( getvar(particles, :vx,     :km_s), getvar(particles, :mass   ));
stats_clumps    = wstat( getvar(clumps,    :peak_x, :kpc ), getvar(clumps,    :mass_cl))  ;

In [50]:
propertynames(stats_gas)

(:mean, :median, :std, :skewness, :kurtosis, :min, :max)

In [51]:
println( "Gas        <vx>_allcells     : ",  stats_gas.mean,       " km/s (mass weighted)" )
println( "Particles  <vx>_allparticles : ",  stats_particles.mean, " km/s (mass weighted)" )
println( "Clumps <peak_x>_allclumps    : ",  stats_clumps.mean,    " kpc  (mass weighted)" )

Gas        <vx>_allcells     : -1.1999253584798235 km/s (mass weighted)
Particles  <vx>_allparticles : -11.623422700314565 km/s (mass weighted)
Clumps <peak_x>_allclumps    : 23.135765457064576 kpc  (mass weighted)


In [52]:
println( "Gas        min/max_allcells     : ",  stats_gas.min,      "/", stats_gas.max,       " km/s" )
println( "Particles  min/max_allparticles : ",  stats_particles.min,"/", stats_particles.max, " km/s" )
println( "Clumps     min/max_allclumps    : ",  stats_clumps.min,   "/", stats_clumps.max,    " Msol/pc^3" )

Gas        min/max_allcells     : -676.5464963488397/894.9181733956399 km/s
Particles  min/max_allparticles : -874.6440509326601/670.7956741234592 km/s
Clumps     min/max_allclumps    : 10.29199219000667/38.17382813002474 Msol/pc^3


For the average of the gas-density use volume weighting:

In [53]:
stats_gas = wstat( getvar(gas, :rho, :g_cm3), weight=getvar(gas, :volume) );

In [54]:
println( "Gas  <rho>_allcells : ",  stats_gas.mean,  " g/cm^3 (volume weighted)" )

Gas  <rho>_allcells : 1.8958545012297404e-26 g/cm^3 (volume weighted)


## Helpful Functions


Get the x,y,z positions of every cell relative to a given center:

In [55]:
x,y,z = getpositions(gas, :kpc, center=[24.,24.,24.], center_unit=:kpc); # returns a Tuple of 3 arrays

The box-center can be calculated automatically:

In [56]:
x,y,z = getpositions(gas, :kpc, center=[:boxcenter]);

In [57]:
[x y z] # preview of the output

849332×3 Matrix{Float64}:
 -23.625   -23.625    -23.625
 -23.625   -23.625    -22.875
 -23.625   -23.625    -22.125
 -23.625   -23.625    -21.375
 -23.625   -23.625    -20.625
 -23.625   -23.625    -19.875
 -23.625   -23.625    -19.125
 -23.625   -23.625    -18.375
 -23.625   -23.625    -17.625
 -23.625   -23.625    -16.875
 -23.625   -23.625    -16.125
 -23.625   -23.625    -15.375
 -23.625   -23.625    -14.625
   ⋮                  
  16.0313    3.84375    0.09375
  16.0313    3.84375    0.28125
  16.0313    3.84375    0.46875
  16.0313    3.84375    0.65625
  16.0313    4.03125   -0.65625
  16.0313    4.03125   -0.46875
  16.0313    4.03125   -0.28125
  16.0313    4.03125   -0.09375
  16.0313    4.03125    0.09375
  16.0313    4.03125    0.28125
  16.0313    4.03125    0.46875
  16.0313    4.03125    0.65625

Get the extent of the dataset-domain:

In [58]:
getextent(gas) # returns Tuple of (xmin, xmax), (ymin ,ymax ), (zmin ,zmax )

((0.0, 48.0), (0.0, 48.0), (0.0, 48.0))

Get the extent relative to a given center:

In [59]:
getextent(gas, center=[:boxcenter])

((-24.0, 24.0), (-24.0, 24.0), (-24.0, 24.0))

Get simulation time in code unit oder physical unit

In [60]:
gettime(info)

39.9019537349027

In [61]:
gettime(info, :Myr)

594.9774920106152

In [62]:
gettime(gas, :Myr)

594.9774920106152

## Summary

This tutorial demonstrated MERA's powerful capabilities for basic calculations and statistical analysis across multi-physics simulation data. The unified interface enables seamless analysis of hydro, particle, and clump data with consistent syntax and automatic unit handling.

### Key Takeaways

#### Essential Functions Covered
- **`msum`**: Mass summation with automatic unit conversion
- **`center_of_mass`**: Mass-weighted spatial averaging
- **`bulk_velocity`**: Mass-weighted velocity centroids
- **`wstat`**: Comprehensive statistical analysis with weighting options
- **`getvar`**: Flexible variable extraction with unit conversion
